# BDA
## Consultas SQL en MariaDB, PostgreSQL y SQLite


---

## Instalación de dependencias

In [ ]:
# Instalar dependencias necesarias
#!pip install pandas mysql-connector-python psycopg2-binary

## Importar librerías

In [ ]:
import pandas as pd
import warnings
# pandas avisa de que solo prueba oficialmente conexiones SQLAlchemy; con sqlite3,
# mysql.connector y psycopg2 funciona igual, así que solo se oculta ese aviso
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy', category=UserWarning)

print("Librerías importadas correctamente")

## Configuración de Conexión



In [ ]:
# ===========================================================================
# CONFIGURACIÓN - Edita estos valores
# ===========================================================================

# Selecciona el tipo de base de datos: 'sqlite', 'mariadb' o 'postgres'
TIPO_BD = 'postgres'

# Configuración para SQLite
SQLITE_FILE = '../dbs/docencia.db'

# Configuración para MariaDB/MySQL
MARIADB_HOST = 'localhost'
MARIADB_PORT = 3306
MARIADB_DATABASE = 'docencia'
MARIADB_USER = 'masteruser'
MARIADB_PASSWORD = 'masterpassword'

# Configuración para PostgreSQL
POSTGRES_HOST = 'localhost'
POSTGRES_PORT = 5432
POSTGRES_DATABASE = 'master'
POSTGRES_USER = 'masteruser'
POSTGRES_PASSWORD = 'masterpassword'
POSTGRES_SCHEMA = 'docencia'

print(f"Configuración lista para: {TIPO_BD.upper()}")

## Conectar a la Base de Datos

In [ ]:
# Variable global para la conexión
conn = None

print("Conectando a la base de datos...\n")

if TIPO_BD == 'sqlite':
    import sqlite3
    conn = sqlite3.connect(SQLITE_FILE)
    print(f"Conectado a SQLite: {SQLITE_FILE}")
    
elif TIPO_BD == 'mariadb':
    import mysql.connector
    conn = mysql.connector.connect(
        host=MARIADB_HOST,
        port=MARIADB_PORT,
        database=MARIADB_DATABASE,
        user=MARIADB_USER,
        password=MARIADB_PASSWORD
    )
    cursor = conn.cursor()
    cursor.execute(f"USE {MARIADB_DATABASE}")
    cursor.close()
    print(f"Conectado a MariaDB/MySQL")
    print(f"Host: {MARIADB_HOST}:{MARIADB_PORT}")
    print(f"Base de datos: {MARIADB_DATABASE}")
    
elif TIPO_BD == 'postgres':
    import psycopg2
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DATABASE,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        options=f'-c search_path={POSTGRES_SCHEMA}'
    )
    print(f"Conectado a PostgreSQL")
    print(f"Host: {POSTGRES_HOST}:{POSTGRES_PORT}")
    print(f"Base de datos: {POSTGRES_DATABASE}")
    print(f"Esquema: {POSTGRES_SCHEMA}")
    
else:
    print(f"ERROR: Tipo de base de datos '{TIPO_BD}' no válido")
    print("Usa: 'sqlite', 'mariadb' o 'postgres'")

## CONSULTAS

In [ ]:
# Definir consultas

consultas = [
"""
SELECT * 
FROM Departamento;
""",
"""
SELECT COUNT(*)
FROM Docencia;
""",
"""
SELECT nombre
FROM Profesor
WHERE edad < 45;
"""
]

def run_query(query, connection, verbose=True, export=False):
    
    if verbose:
        print("Ejecutando consulta...")
        print("="*70)
        print(f"SQL:\n{query}")
        print("="*70)

    # Ejecutar consulta
    try:
        if TIPO_BD == 'postgres':
            cursor = connection.cursor()
            cursor.execute(f"SET search_path TO {POSTGRES_SCHEMA}")
            cursor.close()

        df = pd.read_sql(query, connection)
    except Exception as e:
        # Se informa del error y se sigue con la siguiente consulta de la lista
        print(f"ERROR al ejecutar la consulta: {e}")
        return None

    # Mostrar resultados
    if verbose:
        print(f"TOTAL DE FILAS: {len(df)}")
        print("="*70)
        
        print(f"\nINFORMACIÓN DEL DATAFRAME:")
        print(f"  - Columnas: {list(df.columns)}")
        print(f"  - Dimensiones: {df.shape}")
        print(f"  - Tipos de datos:")
        for col, dtype in df.dtypes.items():
            print(f"     {col}: {dtype}")
    
        print(f"\nPRIMERAS 20 FILAS:")
        print("="*70)
        display(df.head(20))


    # Exportar a TSV
    if export:
        filename = 'result.tsv'
        df.to_csv(filename, sep='\t', index=False)
        print(f"Datos exportados a: {filename}")
        print(f"  - Formato: TSV (Tab-Separated Values)")
        print(f"  - Filas exportadas: {len(df)}")
        print(f"  - Columnas exportadas: {len(df.columns)}")

    print("\n\n")
    return df


for q in consultas:
    df_result = run_query(q, conn)

## Cerrar Conexión

Cierra la conexión a la base de datos cuando termines:

In [ ]:
if conn is not None:
    conn.close()
    print(f"Conexión a {TIPO_BD.upper()} cerrada correctamente")